# SQ-GAN coordinated worker: awgn/ssm

Only the assigned stage may train. Shared parent artifacts are validated and selected by full-epoch progress. Joint training requires both 150-epoch prerequisites. The author architecture, batch size, dataset exposure and precision remain unchanged.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, time, hashlib, shutil, platform

RUN_MODE = "full"  # full training: 150 epochs x 3 stages x 2 channels
EPOCHS_PER_STAGE = 150
SESSION_HOURS = 9.5
SESSION_DEADLINE_UNIX = time.time() + SESSION_HOURS * 3600
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"  # legacy author torch.load behavior
STOP_AFTER_STAGE_EPOCHS = 150  # preserve the total stage target; session deadline handles chunking
CHANNELS = ["awgn", "rayleigh"]
SNR_DB = 10.0
PERFECT_CSI = True  # explicit Rayleigh adaptation assumption
IMAGE_HEIGHT = 256
IMAGE_WIDTH = 512
BATCH_SIZE = 1  # preserve author config

REPO = Path("/tmp/SQ-GAN")  # source, top-k checkpoints and images stay outside exported output
OUTPUT = Path("/tmp/sqgan_merge_validation")
DIV2K_TRAIN = Path("/kaggle/input/notebooks/jagan028/div2k-dataset-generation-for-isr/Training/HR/hr_images")
DIV2K_VAL = Path("/kaggle/input/notebooks/jagan028/div2k-dataset-generation-for-isr/Validation/HR/hr_images")
KODAK_ROOT = Path("/kaggle/input/datasets/sherylmehta/kodak-dataset")
UPSTREAM_URL = "https://github.com/frapez1/SQ-GAN.git"
UPSTREAM_COMMIT = "0e5cdd7c0240b89749dad684645c53f783ac24cc"
BASE_CONFIG = "config/sqgan_cityscapes.yml"
OUTPUT.mkdir(parents=True, exist_ok=True)
print(json.dumps({"python":sys.version,"platform":platform.platform(),"run_mode":RUN_MODE,"upstream_commit":UPSTREAM_COMMIT}, indent=2))

STAGE_PLAN = {'awgn': ['ssm']}
WORKER_STAGE = 'awgn/ssm'


In [ ]:
# Install the legacy Lightning generation required by the authors' two-optimizer training_step.
# Do not replace this with Lightning 2.x: it removed optimizer_idx and the old Trainer CLI APIs.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pytorch-lightning==1.9.5", "omegaconf==2.3.0",
                "torchmetrics==0.11.4", "einops", "easydict", "transformers==4.45.2"], check=True)
if not REPO.exists():
    subprocess.run(["git", "clone", UPSTREAM_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", UPSTREAM_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True)
import numpy as np
if not hasattr(np, "Inf"):
    np.Inf = np.inf  # runtime compatibility for NumPy 2; does not modify upstream source
for p in (REPO / "adaptations", REPO / "adaptations" / "sqgan_div2k_channel"):
    p.mkdir(parents=True, exist_ok=True)
(REPO / "adaptations" / "__init__.py").write_text("", encoding="utf-8")
(REPO / "adaptations" / "sqgan_div2k_channel" / "__init__.py").write_text("", encoding="utf-8")


In [ ]:
# Keep the adaptation code versioned in this notebook; the cloned author checkout stays untouched.
CHANNEL_SOURCE = 'from __future__ import annotations\n\nimport torch\nfrom torch import nn\n\n\nclass ComplexLatentChannel(nn.Module):\n    """Per-example unit-power complex AWGN or flat Rayleigh channel.\n\n    Input follows the author VQ module layout [batch, real_feature_dim, tokens].\n    A 256-D real code vector is serialized as 128 complex channel uses.\n    This is a DIV2K/AWGN/Rayleigh adaptation, not the original SQ-GAN pipeline.\n    """\n    def __init__(self, channel: str, snr_db: float = 10.0, perfect_csi: bool = True):\n        super().__init__()\n        self.channel = channel.lower()\n        if self.channel not in {"awgn", "rayleigh"}:\n            raise ValueError(f"channel must be awgn or rayleigh, got {channel!r}")\n        self.snr_db = float(snr_db)\n        self.perfect_csi = bool(perfect_csi)\n\n    def forward(self, z: torch.Tensor) -> torch.Tensor:\n        if z.ndim != 3:\n            raise ValueError(f"Expected [B,C,N] latent tensor, got {tuple(z.shape)}")\n        b, c, n = z.shape\n        if c % 2:\n            raise ValueError(f"Real latent dimension must be even, got {c}")\n        pairs = z.transpose(1, 2).contiguous().reshape(b, n, c // 2, 2)\n        power = pairs.square().sum(dim=-1).mean(dim=(1, 2), keepdim=True).clamp_min(1e-12)\n        scaled = pairs / power.sqrt().unsqueeze(-1)\n        x = torch.complex(scaled[..., 0], scaled[..., 1])\n        noise_power = 10.0 ** (-self.snr_db / 10.0)\n        sigma = (noise_power / 2.0) ** 0.5\n        noise = torch.complex(torch.randn_like(x.real), torch.randn_like(x.real)) * sigma\n        if self.channel == "rayleigh":\n            h = torch.complex(torch.randn_like(x.real), torch.randn_like(x.real)) / (2.0 ** 0.5)\n            y = h * x + noise\n            if self.perfect_csi:\n                x_hat = y * h.conj() / h.abs().square().clamp_min(1e-8)\n            else:\n                x_hat = y\n        else:\n            x_hat = x + noise\n        received = torch.stack((x_hat.real, x_hat.imag), dim=-1) * power.sqrt().unsqueeze(-1)\n        return received.reshape(b, n, c).transpose(1, 2).contiguous()\n\n\nclass ChannelizedVectorQuantizer(nn.Module):\n    """Keep author VQ and insert a differentiable channel after its STE output."""\n    def __init__(self, quantizer_config: dict, channel: str, snr_db: float = 10.0,\n                 perfect_csi: bool = True):\n        super().__init__()\n        from utils.utils import instantiate_from_config\n        self.quantizer = instantiate_from_config(quantizer_config)\n        self.channel = ComplexLatentChannel(channel, snr_db, perfect_csi)\n\n    def forward(self, *args, **kwargs):\n        quantized, commitment_loss, info = self.quantizer(*args, **kwargs)\n        return self.channel(quantized), commitment_loss, info\n'
DATASET_SOURCE = 'from __future__ import annotations\n\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch.utils.data import Dataset\nfrom torchvision import transforms\n\n\nclass DIV2KWithPseudoLabels(Dataset):\n    """Paired RGB and 19-class Cityscapes-ID pseudo-labels at author\'s 2:1 geometry."""\n    def __init__(self, image_root: str, label_root: str, size=(256, 512), train: bool = True):\n        self.image_root = Path(image_root)\n        self.label_root = Path(label_root)\n        if isinstance(size, int):\n            self.height = self.width = int(size)\n        else:\n            self.height, self.width = map(int, size)\n        self.train = bool(train)\n        exts = {".png", ".jpg", ".jpeg"}\n        images = sorted(p for p in self.image_root.rglob("*") if p.suffix.lower() in exts)\n        if not images:\n            raise FileNotFoundError(f"No images under {self.image_root}")\n        self.pairs = []\n        missing = []\n        for image in images:\n            mask = self.label_root / f"{image.stem}.png"\n            if mask.is_file():\n                self.pairs.append((image, mask))\n            else:\n                missing.append(str(mask))\n        if missing:\n            raise FileNotFoundError(f"Missing {len(missing)} pseudo masks; first: {missing[0]}")\n        self.image_norm = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))\n\n    def __len__(self):\n        return len(self.pairs)\n\n    def _resize_cover(self, image, mask):\n        scale = max(self.width / image.width, self.height / image.height)\n        new_size = (max(self.width, round(image.width * scale)), max(self.height, round(image.height * scale)))\n        image = image.resize(new_size, Image.Resampling.BICUBIC)\n        mask = mask.resize(new_size, Image.Resampling.NEAREST)\n        return image, mask\n\n    def __getitem__(self, idx):\n        image_path, mask_path = self.pairs[idx]\n        image = Image.open(image_path).convert("RGB")\n        mask = Image.open(mask_path).convert("L")\n        image, mask = self._resize_cover(image, mask)\n        if self.train:\n            x = int(np.random.randint(0, image.width - self.width + 1))\n            y = int(np.random.randint(0, image.height - self.height + 1))\n        else:\n            x = (image.width - self.width) // 2\n            y = (image.height - self.height) // 2\n        box = (x, y, x + self.width, y + self.height)\n        image, mask = image.crop(box), mask.crop(box)\n        if self.train and np.random.rand() < 0.5:\n            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)\n            mask = mask.transpose(Image.Transpose.FLIP_LEFT_RIGHT)\n        image_tensor = self.image_norm(transforms.ToTensor()(image))\n        ids = torch.from_numpy(np.asarray(mask, dtype=np.uint8).copy()).long()\n        if torch.any((ids != 255) & (ids > 18)):\n            raise ValueError(f"Expected Cityscapes IDs [0,18] or 255: {mask_path}")\n        ids[ids == 255] = 19\n        semantic = torch.nn.functional.one_hot(ids, num_classes=20).permute(2, 0, 1).float()\n        label_ids = ids.clone()\n        label_ids[label_ids == 19] = 0\n        label = label_ids.unsqueeze(0).float()\n        return {"image": image_tensor, "label": label, "semantic": semantic, "filename": image_path.name}\n'
ADAPTER = REPO / "adaptations" / "sqgan_div2k_channel"
(ADAPTER / "channel.py").write_text(CHANNEL_SOURCE, encoding="utf-8")
(ADAPTER / "dataset.py").write_text(DATASET_SOURCE, encoding="utf-8")







# BEGIN SESSION ORCHESTRATION
COMPAT_DIR = Path("/tmp/sqgan_runtime")
COMPAT_DIR.mkdir(parents=True, exist_ok=True)
(COMPAT_DIR / 'session_control.py').write_text('"""Notebook orchestration for the pinned author Lightning 1.9 training script."""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport os\nimport time\nfrom pathlib import Path\n\nimport pytorch_lightning as pl\nfrom pytorch_lightning.callbacks import Callback\n\n\nclass SessionControl(Callback):\n    def __init__(self):\n        super().__init__()\n        self.deadline = float(os.environ.get("SQGAN_SESSION_DEADLINE_UNIX", "inf"))\n        self.epoch_limit = int(os.environ.get("SQGAN_STOP_AFTER_STAGE_EPOCHS", "150"))\n        self.epoch_started = None\n\n    def on_train_start(self, trainer, pl_module):\n        expected = json.loads(os.environ.get("SQGAN_EXPECTED_RESUME", "null"))\n        rates = [[g["lr"] for g in opt.param_groups] for opt in trainer.optimizers]\n        actual = {"epoch_index": int(trainer.current_epoch), "global_step": int(trainer.global_step), "optimizer_learning_rates": rates}\n        if expected:\n            assert actual["global_step"] == expected["global_step"], (actual, expected)\n            assert len(trainer.optimizers) == expected["optimizer_states"], (actual, expected)\n            assert len(trainer.lr_scheduler_configs) == expected["lr_schedulers"], (actual, expected)\n            assert actual["epoch_index"] >= expected["saved_epoch_index"], (actual, expected)\n            for actual_groups, expected_groups in zip(rates, expected["optimizer_learning_rates"]):\n                assert len(actual_groups) == len(expected_groups)\n                assert all(math.isclose(a, b, rel_tol=1e-7, abs_tol=1e-12) for a, b in zip(actual_groups, expected_groups)), (actual, expected)\n            actual["resume_verified"] = True\n        else:\n            actual["resume_verified"] = False\n        if trainer.is_global_zero:\n            print("TRAINING STATE VERIFIED", json.dumps(actual), flush=True)\n            root = os.environ.get("SQGAN_STAGE_AUDIT_DIR")\n            if root:\n                destination = Path(root)\n                destination.mkdir(parents=True, exist_ok=True)\n                (destination / "resume_verified.json").write_text(json.dumps(actual, indent=2))\n\n    def on_train_epoch_start(self, trainer, pl_module):\n        self.epoch_started = time.time()\n\n    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, *args, **kwargs):\n        if trainer.is_global_zero and batch_idx % 100 == 0:\n            print("TRAIN PROGRESS", json.dumps({"epoch_index":int(trainer.current_epoch),"batch_index":batch_idx,"global_step":int(trainer.global_step),"training_batches":trainer.num_training_batches}), flush=True)\n\n    def on_train_epoch_end(self, trainer, pl_module, *args, **kwargs):\n        elapsed = time.time() - self.epoch_started if self.epoch_started else 0.0\n        completed = int(trainer.current_epoch) + 1\n        # Leave enough time for checkpoint serialization, mirroring and output upload.\n        local_stop = completed >= self.epoch_limit or time.time() + max(600, 1.5 * elapsed) >= self.deadline\n        stop = trainer.strategy.reduce_boolean_decision(local_stop, all=False)\n        trainer.should_stop = trainer.should_stop or stop\n        if trainer.is_global_zero:\n            record = {"completed_epochs": completed, "global_step": int(trainer.global_step), "epoch_seconds": elapsed, "session_seconds_remaining": max(0, self.deadline-time.time())}\n            for key, value in trainer.callback_metrics.items():\n                if getattr(value, "numel", lambda: 0)() == 1:\n                    record[key] = float(value.detach().cpu())\n            root = os.environ.get("SQGAN_STAGE_AUDIT_DIR")\n            if root:\n                with (Path(root) / "training_history.jsonl").open("a") as stream:\n                    stream.write(json.dumps(record) + "\\n")\n            print("EPOCH AUDIT", json.dumps(record), flush=True)\n            if stop:\n                print("NORMAL SESSION STOP REQUESTED at completed stage epoch", completed, flush=True)\n\n\ndef install():\n    if getattr(pl.Trainer, "_sqgan_session_control_installed", False):\n        return\n    original = pl.Trainer.fit\n\n    def fit(self, *args, **kwargs):\n        self.callbacks.append(SessionControl())\n        return original(self, *args, **kwargs)\n\n    pl.Trainer.fit = fit\n    pl.Trainer._sqgan_session_control_installed = True\n', encoding="utf-8")
(COMPAT_DIR / 'check_resume_contract.py').write_text('"""Verify Lightning checkpoint continuation on a tiny CPU model, not SQ-GAN."""\nimport json\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nif not hasattr(np, "Inf"):\n    np.Inf = np.inf\nos.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"\n\nimport torch\nimport pytorch_lightning as pl\nfrom torch.utils.data import DataLoader, TensorDataset\nfrom session_control import install\n\n\nclass Probe(pl.LightningModule):\n    def __init__(self):\n        super().__init__()\n        self.generator = torch.nn.Linear(2, 1)\n        self.discriminator = torch.nn.Linear(2, 1)\n\n    def training_step(self, batch, batch_idx, optimizer_idx):\n        layer = self.generator if optimizer_idx == 0 else self.discriminator\n        return layer(batch[0]).square().mean()\n\n    def configure_optimizers(self):\n        opts = [torch.optim.Adam(layer.parameters(), lr=0.01) for layer in [self.generator, self.discriminator]]\n        return opts, [{"scheduler": torch.optim.lr_scheduler.StepLR(opt, step_size=2, gamma=0.5), "interval": "step"} for opt in opts]\n\n\ndef main(root):\n    root = Path(root)\n    root.mkdir(parents=True, exist_ok=True)\n    install()\n    os.environ["SQGAN_SESSION_DEADLINE_UNIX"] = "inf"\n    os.environ["SQGAN_STOP_AFTER_STAGE_EPOCHS"] = "2"\n    os.environ["SQGAN_EXPECTED_RESUME"] = "null"\n    os.environ["SQGAN_STAGE_AUDIT_DIR"] = str(root / "first")\n    loader = DataLoader(TensorDataset(torch.ones(8, 2)), batch_size=2)\n    def trainer():\n        return pl.Trainer(accelerator="cpu", devices=1, max_epochs=4, logger=False, enable_checkpointing=False, enable_progress_bar=False, enable_model_summary=False)\n    first = trainer()\n    first.fit(Probe(), loader)\n    checkpoint = root / "probe.ckpt"\n    first.save_checkpoint(checkpoint)\n    raw = torch.load(checkpoint, map_location="cpu", weights_only=False)\n    assert raw["epoch"] == 2 or raw["epoch"] == 1\n    expected = {"saved_epoch_index": raw["epoch"], "global_step": raw["global_step"], "optimizer_states":len(raw["optimizer_states"]),"lr_schedulers":len(raw["lr_schedulers"]),"optimizer_learning_rates":[[g["lr"] for g in opt["param_groups"]] for opt in raw["optimizer_states"]]}\n    os.environ["SQGAN_EXPECTED_RESUME"] = json.dumps(expected)\n    os.environ["SQGAN_STOP_AFTER_STAGE_EPOCHS"] = "4"\n    os.environ["SQGAN_STAGE_AUDIT_DIR"] = str(root / "second")\n    second = trainer()\n    second.fit(Probe(), loader, ckpt_path=str(checkpoint))\n    assert second.current_epoch == 4, second.current_epoch\n    assert second.global_step > raw["global_step"]\n    os.environ["SQGAN_EXPECTED_RESUME"] = "null"\n    os.environ["SQGAN_SESSION_DEADLINE_UNIX"] = "0"\n    os.environ["SQGAN_STAGE_AUDIT_DIR"] = str(root / "deadline")\n    timed = trainer()\n    timed.fit(Probe(), loader)\n    assert timed.current_epoch == 1, timed.current_epoch\n    timed.save_checkpoint(root / "deadline_stop.ckpt")\n    timed_state = torch.load(root / "deadline_stop.ckpt", map_location="cpu", weights_only=False)\n    assert len(timed_state["optimizer_states"]) == len(timed_state["lr_schedulers"]) == 2\n    report = {"result":"PASS", "initial_global_step":raw["global_step"],"resumed_final_global_step":second.global_step,"resumed_final_epoch":second.current_epoch,"optimizer_and_scheduler_states_restored":True,"deadline_stop_and_checkpoint_verified":True}\n    (root / "resume_contract_pass.json").write_text(json.dumps(report, indent=2))\n    print("RESUME CONTRACT PASS", json.dumps(report), flush=True)\n\n\nif __name__ == "__main__":\n    import sys\n    main(sys.argv[1])\n', encoding="utf-8")


(COMPAT_DIR/"stage_artifacts.py").write_text('"""Validate and combine independent stage artifacts, never average checkpoints."""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport shutil\n\nPIN=\'0e5cdd7c0240b89749dad684645c53f783ac24cc\'\nCHANNELS=(\'awgn\',\'rayleigh\')\nMODES=(\'ssm\',\'img\',\'all\')\nSTAGES=tuple((c,m) for c in CHANNELS for m in MODES)\nPARTS={\'ssm\':(\'encoder_semantic\',\'decoder_semantic\',\'masker_semantic\',\'demasker_semantic\',\'quantize_semantic\'),\n       \'img\':(\'encoder\',\'decoder\',\'masker\',\'demasker\',\'quantize\')}\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda:stream.read(8*1024*1024),b\'\'): h.update(chunk)\n    return h.hexdigest()\n\ndef select_stages(records):\n    """Strictly pick greatest progress; equal-progress conflicts require review."""\n    selected={}\n    for row in records:\n        key=(row[\'channel\'],row[\'mode\'])\n        assert key in STAGES\n        old=selected.get(key)\n        if old is None or row[\'global_step\']>old[\'global_step\']:\n            selected[key]=row\n        elif row[\'global_step\']==old[\'global_step\']:\n            assert row[\'sha256\']==old[\'sha256\'], f\'Conflicting checkpoints at equal progress: {key}\'\n    return selected\n\ndef ready_stages(inventory,leased=()):\n    leased=set(leased)\n    result=[]\n    for key in STAGES:\n        if key in leased or inventory.get(key,{}).get(\'completed_epochs\',0)>=150: continue\n        c,m=key\n        if m==\'all\' and not all(inventory.get((c,p),{}).get(\'completed_epochs\',0)>=150 for p in (\'ssm\',\'img\')):\n            continue\n        result.append(key)\n    # Preserve partial-stage work before starting untouched stages.\n    return sorted(result,key=lambda k:(-inventory.get(k,{}).get(\'completed_epochs\',0),STAGES.index(k)))\n\ndef validate_checkpoint(path,channel,mode):\n    import torch\n    from omegaconf import OmegaConf\n    raw=torch.load(path,map_location=\'cpu\',weights_only=False)\n    step=int(raw[\'global_step\'])\n    assert step>0 and step%800==0, f\'Checkpoint is not at a full epoch boundary: {step}\'\n    epochs=step//800  # 800 images / (1 image x 2 ranks), two optimizers/batch\n    assert 0<epochs<=150\n    assert int(raw[\'epoch\']) in {epochs-1,epochs}\n    assert len(raw[\'optimizer_states\'])==len(raw[\'lr_schedulers\'])==2\n    state=raw[\'state_dict\']\n    required=PARTS[\'ssm\']+PARTS[\'img\'] if mode==\'all\' else PARTS[\'ssm\' if mode==\'ssm\' else \'img\']\n    for prefix in required:\n        assert any(k.startswith(prefix+\'.\') for k in state), (mode,prefix)\n    # Matching source configuration is mandatory for exposure and geometry.\n    return raw,{\'channel\':channel,\'mode\':mode,\'saved_epoch_index\':int(raw[\'epoch\']),\n                \'completed_epochs\':epochs,\'global_step\':step,\'target_epochs\':150,\n                \'optimizer_states\':2,\'lr_schedulers\':2,\'sha256\':sha256(path),\n                \'status\':\'complete\' if epochs==150 else \'partial\'}\n\ndef merge_artifacts(input_root,output,repo):\n    import torch\n    from omegaconf import OmegaConf\n    output,repo=Path(output),Path(repo)\n    roots=sorted(p for p in Path(input_root).rglob(\'sqgan_div2k_channel_results\')\n                 if (p/\'run_manifest.json\').is_file())\n    assert roots,\'No successful SQ-GAN artifact input supplied\'\n    candidates=[]\n    label_protocol=None\n    for root in roots:\n        manifest=json.loads((root/\'run_manifest.json\').read_text())\n        assert manifest[\'upstream_commit\']==PIN and manifest[\'epochs_per_stage\']==150\n        assert float(manifest[\'snr_db\'])==10.0\n        assert set(manifest[\'channels\'])==set(CHANNELS)\n        metadata=json.loads((root/\'pseudo_label_metadata.json\').read_text())\n        if label_protocol is None: label_protocol=metadata\n        assert metadata==label_protocol,\'Pseudo-label protocol differs across workers\'\n        for stage in manifest[\'stages\']:\n            c,m=stage[\'channel\'],stage[\'mode\']\n            assert (c,m) in STAGES\n            path=root/stage[\'checkpoint_relative\']\n            assert path.resolve().is_relative_to(root.resolve()) and path.is_file()\n            cfg=root/f\'config_{c}_{m}.yaml\'\n            config=OmegaConf.load(cfg)\n            assert config.data.params.batch_size==1\n            assert list(config.data.params.train.params.size)==[256,512]\n            for k in (\'vqconfig\',\'vqconfig_semantic\'):\n                params=config.model.params[k].params\n                assert params.channel==c and float(params.snr_db)==10.0 and params.perfect_csi\n            raw,info=validate_checkpoint(path,c,m)\n            del raw\n            info.update(source_root=str(root),source_path=str(path),config=str(cfg))\n            candidates.append(info)\n    chosen=select_stages(candidates)\n    output.mkdir(parents=True,exist_ok=True)\n    # Shared data are copied once; mutable stage files are selected individually.\n    first=roots[0]\n    for name in (\'pseudo_labels\',\'pseudo_label_metadata.json\'):\n        src=first/name\n        if src.is_dir(): shutil.copytree(src,output/name,dirs_exist_ok=True)\n        else: shutil.copy2(src,output/name)\n    state_dir=output/\'stage_state\'; state_dir.mkdir(exist_ok=True)\n    merge_records=[]\n    for (c,m),row in chosen.items():\n        folder=repo/\'logs\'/f\'coordinated_sqgan_{c}_{m}_150ep\'\n        (folder/\'checkpoints\').mkdir(parents=True,exist_ok=True)\n        (folder/\'configs\').mkdir(exist_ok=True)\n        shutil.copy2(row[\'source_path\'],folder/\'checkpoints/last.ckpt\')\n        shutil.copy2(row[\'config\'],folder/\'configs/project.yaml\')\n        shutil.copy2(row[\'config\'],output/f\'config_{c}_{m}.yaml\')\n        exported=output/\'upstream_logs\'/folder.name\n        shutil.copytree(folder,exported,dirs_exist_ok=True)\n        marker={k:v for k,v in row.items() if k not in {\'source_root\',\'source_path\',\'config\'}}\n        marker[\'checkpoint_relative\']=str((exported/\'checkpoints/last.ckpt\').relative_to(output))\n        marker[\'bytes\']=Path(row[\'source_path\']).stat().st_size\n        (state_dir/f\'{c}_{m}.json\').write_text(json.dumps(marker,indent=2))\n        audit=Path(row[\'source_root\'])/\'audit\'/f\'{c}_{m}\'\n        if audit.exists(): shutil.copytree(audit,output/\'audit\'/f\'{c}_{m}\',dirs_exist_ok=True)\n        if row[\'completed_epochs\']==150:\n            if m in PARTS:\n                raw=torch.load(row[\'source_path\'],map_location=\'cpu\',weights_only=False)\n                parts=output/c/\'components\'; parts.mkdir(parents=True,exist_ok=True)\n                for name in PARTS[m]:\n                    prefix=name+\'.\'\n                    tensors={k[len(prefix):]:v for k,v in raw[\'state_dict\'].items() if k.startswith(prefix)}\n                    assert tensors\n                    torch.save(tensors,parts/(name+\'.ckpt\'))\n                del raw\n            else:\n                (output/c).mkdir(exist_ok=True)\n                (output/c/\'final_checkpoint.json\').write_text(json.dumps(marker,indent=2))\n        merge_records.append(marker)\n    report={\'upstream_commit\':PIN,\'selection\':\'highest validated full-epoch global_step per stage; no weight averaging\',\n            \'stages\':merge_records,\'input_roots\':[str(p) for p in roots]}\n    (output/\'merge_report.json\').write_text(json.dumps(report,indent=2))\n    print(\'STAGE ARTIFACT MERGE VERIFIED\',json.dumps(report),flush=True)\n    return report\n')
sys.path.insert(0,str(COMPAT_DIR))
from stage_artifacts import merge_artifacts
merge_artifacts(Path("/kaggle/input"),OUTPUT,REPO)


In [ ]:
from stage_artifacts import sha256
report=json.loads((OUTPUT/'merge_report.json').read_text())
assert len(report['stages'])==1
row=report['stages'][0]
assert row['completed_epochs']==98 and row['global_step']==78400
original=next(Path('/kaggle/input').rglob('sqgan_div2k_channel_results'))/'upstream_logs/10-19T17-22-21_sqgan_awgn_ssm_150ep/checkpoints/last.ckpt'
merged=OUTPUT/row['checkpoint_relative']
assert sha256(original)==sha256(merged)==row['sha256']
result={'source_and_merged_checkpoint_equal':True,'completed_epochs':98,'global_step':78400,
        'optimizer_states':2,'lr_schedulers':2,'checkpoint_sha256':row['sha256'],'model_updates_performed':0}
out=Path('/kaggle/working/stage_merge_validation'); out.mkdir(exist_ok=True)
(out/'validation.json').write_text(json.dumps(result,indent=2))
print('REAL CHECKPOINT MERGE VALIDATION PASS',json.dumps(result),flush=True)
